In [ ]:
# Cell 1 — self-contained bootstrap
from google.colab import userdata
import torch, os, sys

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_EMAIL = "evenjlinekka@gmail.com"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Evenjlin/deep-space-interference-ml.git"

if not os.path.exists('/content/deep-space-interference-ml'):
    !git clone {REPO_URL} /content/deep-space-interference-ml
%cd /content/deep-space-interference-ml
!git config --global user.email "{GITHUB_EMAIL}"
!git config --global user.name "Evenjlin"
!git remote set-url origin {REPO_URL}
!git pull

!pip install -r requirements.txt -q
sys.path.insert(0, os.getcwd())

In [ ]:
# Cell 2 — config + labeled dataset for time-shift classification
import numpy as np
import torch
from src.channel import SignalConfig, generate_soi, generate_noise, apply_time_shift
from src.model import TimeShiftCNN, complex_to_channels

cfg = SignalConfig(fd_max=0.0)
rng = np.random.default_rng(42)
N_SYMBOLS = 64
SNR_DB = 15.0  # BASE-PAPER FACT (Fig.3/general condition)

def make_shifted_sample(cfg, n_symbols, snr_db, rng):
    s, bits, fd, phi_m = generate_soi(cfg, n_symbols, rng)
    s_shifted, k_shift = apply_time_shift(s, cfg, rng)
    w = generate_noise(len(s), rng)
    rho_snr = 10 ** (snr_db / 10)
    x = s_shifted + w / np.sqrt(rho_snr)
    return x, k_shift

M_TRAIN, M_VAL = 6000, 1500
def build_shift_dataset(n):
    X, y = [], []
    for _ in range(n):
        x, k = make_shifted_sample(cfg, N_SYMBOLS, SNR_DB, rng)
        X.append(complex_to_channels(x))
        y.append(k)
    return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64)

X_train, y_train = build_shift_dataset(M_TRAIN)
X_val, y_val = build_shift_dataset(M_VAL)
print("X_train:", X_train.shape, " y_train classes:", np.unique(y_train))

In [ ]:
# Cell 3 — DEBUG LADDER (mandatory before full training)
model = TimeShiftCNN(cfg.NSPS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# TEST 1: tiny subset forward pass
tiny_x = torch.tensor(X_train[:8], device=device)
out = model(tiny_x)
print("TEST 1: output shape", out.shape, "-- expect (8,", cfg.NSPS+1, ")")
assert out.shape == (8, cfg.NSPS + 1)

# TEST 2: one batch, one grad step
tiny_y = torch.tensor(y_train[:8], device=device)
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss = loss_fn(out, tiny_y)
loss.backward(); opt.step(); opt.zero_grad()
print(f"TEST 2: one grad step OK, loss={loss.item():.4f}")

# TEST 3/4: a few steps on a small slice, loss should trend down
small_x = torch.tensor(X_train[:300], device=device)
small_y = torch.tensor(y_train[:300], device=device)
losses = []
for _ in range(10):
    opt.zero_grad()
    out = model(small_x)
    loss = loss_fn(out, small_y)
    loss.backward(); opt.step()
    losses.append(loss.item())
print("TEST 3/4 losses:", [f"{l:.3f}" for l in losses])
assert losses[-1] < losses[0], "Loss did not decrease -- stop, don't proceed to full training"
print("Debug ladder PASSED.")

In [ ]:
# Cell 4 — full training (fresh model)
def train_shift_cnn(model, X_train, y_train, X_val, y_val, device, 
                      epochs=30, batch_size=128, lr=1e-3, patience=5, seed=42):
    torch.manual_seed(seed)
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.CrossEntropyLoss()
    Xt = torch.tensor(X_train, device=device)
    yt = torch.tensor(y_train, device=device)
    Xv = torch.tensor(X_val, device=device)
    yv = torch.tensor(y_val, device=device)

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val, best_state, patience_ct = float("inf"), None, 0
    n = Xt.shape[0]

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n)
        ep_losses = []
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            opt.zero_grad()
            out = model(Xt[idx])
            loss = loss_fn(out, yt[idx])
            loss.backward(); opt.step()
            ep_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_out = model(Xv)
            val_loss = loss_fn(val_out, yv).item()
            val_acc = (val_out.argmax(dim=1) == yv).float().mean().item()

        train_loss = float(np.mean(ep_losses))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch+1}/{epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        if val_loss < best_val:
            best_val, best_state, patience_ct = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ct += 1
            if patience_ct >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    model.load_state_dict(best_state)
    return model, history

model = TimeShiftCNN(cfg.NSPS)
model, history = train_shift_cnn(model, X_train, y_train, X_val, y_val, device)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history["val_acc"]); ax[1].set_title("Val accuracy")
plt.tight_layout()
plt.savefig("figures/timeshift_cnn_training.png", dpi=150)
plt.show()

import os
os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/timeshift_cnn.pt")

In [ ]:
# Cell 5 — sanity: is the CNN actually better than random guessing?
random_baseline_acc = 1 / (cfg.NSPS + 1)
final_val_acc = history["val_acc"][-1]
print(f"Random-guess baseline accuracy: {random_baseline_acc:.4f}")
print(f"CNN final val accuracy: {final_val_acc:.4f}")
assert final_val_acc > random_baseline_acc * 2, "Barely better than chance -- investigate before proceeding"